# CLS / patch token analysis
Load a nanopath backbone, embed a random slice of the held-out (patient-split) val tiles, then look at how CLS tokens spread on the sphere and how one tile's patch tokens relate to its CLS.

In [ ]:
import os
import random
import sys
from pathlib import Path

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO))

import matplotlib.pyplot as plt
import numpy as np
import pyarrow.parquet as pq
import torch
import torch.nn.functional as F
import yaml
from torch.utils.data import DataLoader, Subset
from torchvision.transforms import v2

from dataloader import TCGATileDataset
from model import ViT, load_pretrained
from train import kde_loss

In [ ]:
CKPT = Path("/data/raymond.biju/nanopath/20260916-xarch_01_clsspec0/latest.pt")  # None = stock DINOv2 weights
WEIGHTS = "model_ema"  # "model_ema" (teacher, what probes use) or "model" (student)
CONFIG = REPO / "configs/baseline.yaml"  # only read when CKPT is None
N_SAMPLES, BATCH, SEED = 2048, 128, 0
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Model loading + val-set inference

In [ ]:
# Rebuild the backbone from a train.py checkpoint (its saved config drives variant + data), or stock DINOv2 when ckpt is None.
def load_model(ckpt=CKPT, weights=WEIGHTS):
    if ckpt is None:
        cfg = yaml.safe_load(os.path.expandvars(CONFIG.read_text()))
        return load_pretrained(ViT(variant=cfg["model"]["type"])).to(device).eval(), cfg
    checkpoint = torch.load(ckpt, map_location="cpu", weights_only=False)
    cfg = checkpoint["config"]
    model = ViT(variant=cfg["model"]["type"])
    model.load_state_dict(checkpoint[weights], strict=True)
    return model.to(device).eval(), cfg


model, cfg = load_model()

In [ ]:
# Val split with deterministic preprocessing: the whole 224px tile, normalized, one view, no local crops.
val_ds = TCGATileDataset({**cfg, "config_path": str(CONFIG)}, is_train=False)
val_ds.global_views, val_ds.local_views = 1, 0
val_ds.global_aug = v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True), v2.Normalize(cfg["data"]["mean"], cfg["data"]["std"])])
sample_idx = random.Random(SEED).sample(range(len(val_ds)), N_SAMPLES)
print(f"val tiles: {len(val_ds):,}  sampled: {N_SAMPLES}")

In [ ]:
# One forward pass over the sampled tiles; keeps L2-normalized CLS (float32) and patch tokens (float16, ~400MB at 2048 tiles) on CPU.
@torch.no_grad()
def embed(model, ds, indices):
    out = {"cls": [], "patches": [], "slide_id": [], "idx": []}
    for batch in DataLoader(Subset(ds, indices), batch_size=BATCH, num_workers=8):
        with torch.autocast(device.type, dtype=torch.bfloat16, enabled=device.type == "cuda"):
            tokens = model(batch["global_views"][:, 0].to(device, non_blocking=True))
        out["cls"].append(F.normalize(tokens["cls"].float(), dim=-1).cpu())
        out["patches"].append(F.normalize(tokens["patches"].float(), dim=-1).half().cpu())
        out["slide_id"].append(batch["slide_id"])
        out["idx"].append(batch["sample_idx"])
    return {k: torch.cat(v) for k, v in out.items()}


emb = embed(model, val_ds, sample_idx)
{k: tuple(v.shape) for k, v in emb.items()}

## CLS spread

In [ ]:
# How CLS tokens fill the sphere: pairwise/NN similarity (NN chord is kde_radius's scale), uniformity, rank, slide clustering, KDE loss.
@torch.no_grad()
def cls_spread(emb, concentration=None, radius=None):
    concentration = cfg["dino"]["kde_concentration"] if concentration is None else concentration
    radius = cfg["dino"].get("kde_radius", 0.0) if radius is None else radius  # older checkpoints predate kde_radius
    x = emb["cls"].to(device)
    off = ~torch.eye(len(x), dtype=torch.bool, device=device)
    cos = x @ x.T
    pair = cos[off]
    nn_cos = cos.masked_fill(~off, -2).amax(1)
    same_slide = (emb["slide_id"][:, None] == emb["slide_id"][None]).to(device) & off
    var = torch.linalg.svdvals(x - x.mean(0)) ** 2
    var = var / var.sum()
    kde = torch.stack([kde_loss(c, concentration, radius) for c in x.split(BATCH) if len(c) == BATCH]).mean()
    stats = {
        "mean_pairwise_cos": pair.mean().item(),
        "std_pairwise_cos": pair.std().item(),
        "mean_nn_cos": nn_cos.mean().item(),
        "median_nn_chord": (2 - 2 * nn_cos).clamp_min(0).sqrt().median().item(),
        "mean_vector_norm": x.mean(0).norm().item(),  # 0 = centred on the sphere, 1 = collapsed to one direction
        "uniformity": torch.log(torch.exp(-2 * (2 - 2 * pair)).mean()).item(),  # lower = more uniform
        "effective_rank": torch.exp(-(var * var.clamp_min(1e-12).log()).sum()).item(),
        "top10_pc_variance": var[:10].sum().item(),
        "same_slide_pairs": int(same_slide.sum().item()),
        "same_slide_cos": cos[same_slide].mean().item() if same_slide.any() else float("nan"),
        "cross_slide_cos": cos[off & ~same_slide].mean().item(),
        f"kde_loss(k={concentration}, r={radius})": kde.item(),
    }

    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    bins = np.linspace(-1, 1, 101)
    axes[0].hist(pair.cpu().numpy(), bins=bins, density=True, alpha=0.6, label="all pairs")
    axes[0].hist(nn_cos.cpu().numpy(), bins=bins, density=True, alpha=0.6, label="nearest neighbour")
    if radius > 0:
        axes[0].axvline(1 - radius ** 2 / 2, color="k", ls="--", label=f"kde_radius {radius}")
    axes[0].set(title="CLS cosine similarity", xlabel="cos")
    axes[0].legend()
    if same_slide.any():
        axes[1].hist(cos[same_slide].cpu().numpy(), bins=bins, density=True, alpha=0.6, label="same slide")
    axes[1].hist(cos[off & ~same_slide].cpu().numpy(), bins=bins, density=True, alpha=0.6, label="cross slide")
    axes[1].set(title="Slide clustering", xlabel="cos")
    axes[1].legend()
    axes[2].plot(var.cpu().numpy(), marker=".")
    axes[2].set(title="Centred CLS variance spectrum", xlabel="component", ylabel="variance fraction", yscale="log")
    plt.tight_layout()
    plt.show()
    return stats


cls_spread(emb)

## Patch tokens vs CLS for one tile

In [ ]:
# One val tile (random unless `idx`): patch-CLS cosine, last-block CLS attention (+ where its mass goes), pre-norm token norms, and patches in a dataset-wide PCA basis.
@torch.no_grad()
def show_patch_tokens(model, ds, emb, idx=None, seed=None, n_background=20000):
    idx = random.Random(seed).randrange(len(ds)) if idx is None else idx
    x = ds[idx]["global_views"].to(device)
    attn_mod, R = model.blocks[-1].attn, model.registers
    captured = {}
    handles = [attn_mod.register_forward_hook(lambda m, inp, out: captured.update(attn_in=inp[0])),
               model.norm.register_forward_hook(lambda m, inp, out: captured.update(pre_norm=inp[0]))]
    tokens = model(x)
    for handle in handles:
        handle.remove()

    # SDPA never materializes weights, so recompute last-block attention from the captured (post-norm1) input.
    t = captured["attn_in"]
    B, N, C = t.shape
    q, k, _ = attn_mod.qkv(t).reshape(B, N, 3, attn_mod.heads, C // attn_mod.heads).permute(2, 0, 3, 1, 4).unbind(0)
    cls_row = ((q @ k.transpose(-2, -1)) / (C // attn_mod.heads) ** 0.5).softmax(-1)[0, :, 0]  # (heads, 1 + R + P): CLS query over every token
    cls_attn = cls_row[:, 1 + R:]
    attn_mass = torch.stack([cls_row[:, 0], cls_row[:, 1:1 + R].sum(-1), cls_attn.sum(-1)], 1).cpu()  # (heads, 3): self / registers / patches
    # High-norm artifact tokens show up before the final LayerNorm, which flattens per-token scale.
    pre_norm = captured["pre_norm"][0].float().norm(dim=-1)
    patch_norm, reg_norm = pre_norm[1 + R:], pre_norm[1:1 + R]
    cls = F.normalize(tokens["cls"][0].float(), dim=-1)
    patches = F.normalize(tokens["patches"][0].float(), dim=-1)
    grid = int(patches.shape[0] ** 0.5)
    cls_sim = patches @ cls

    background = emb["patches"].flatten(0, 1)
    background = background[torch.randperm(len(background), generator=torch.Generator().manual_seed(0))[:n_background]].float().to(device)
    mean = background.mean(0)
    _, _, V = torch.pca_lowrank(background - mean, q=3, center=False)
    bg_pcs, patch_pcs, cls_pc = [((v - mean) @ V).cpu() for v in (background, patches, cls[None])]
    lo, hi = bg_pcs.quantile(0.01, dim=0), bg_pcs.quantile(0.99, dim=0)
    pca_rgb = ((patch_pcs - lo) / (hi - lo)).clamp(0, 1).reshape(grid, grid, 3)

    off = ~torch.eye(len(patches), dtype=torch.bool, device=device)
    var = torch.linalg.svdvals(patches - patches.mean(0)) ** 2
    var = var / var.sum()
    head_attn = cls_attn.mean(0)
    ranks = torch.stack([head_attn.argsort().argsort(), patch_norm.argsort().argsort()]).float()
    stats = {
        "idx": idx,
        "mean_patch_cls_cos": cls_sim.mean().item(),
        "max_patch_cls_cos": cls_sim.max().item(),
        "mean_patch_pairwise_cos": (patches @ patches.T)[off].mean().item(),
        "patch_effective_rank": torch.exp(-(var * var.clamp_min(1e-12).log()).sum()).item(),
        "cls_attn_entropy": -(head_attn / head_attn.sum() * (head_attn / head_attn.sum()).clamp_min(1e-12).log()).sum().item(),  # log(256)=5.55 is uniform
        "cls_attn_self": attn_mass[:, 0].mean().item(),
        "cls_attn_registers": attn_mass[:, 1].mean().item(),
        "cls_attn_patches": attn_mass[:, 2].mean().item(),
        "patch_norm_median": patch_norm.median().item(),
        "patch_norm_max_over_median": (patch_norm.max() / patch_norm.median()).item(),
        "register_norms": [round(v, 1) for v in reg_norm.tolist()],
        "cls_attn_vs_patch_norm_spearman": torch.corrcoef(ranks)[0, 1].item(),  # >0 means CLS reads the high-norm patches
    }

    img = (x[0].cpu() * torch.tensor(cfg["data"]["std"])[:, None, None] + torch.tensor(cfg["data"]["mean"])[:, None, None]).clamp(0, 1).permute(1, 2, 0)
    extent = (0, img.shape[1], img.shape[0], 0)
    fig, axes = plt.subplots(2, 4, figsize=(24, 11))
    axes = axes.flatten()
    axes[0].imshow(img)
    axes[0].set_title(f"val tile {idx}")
    overlays = [("patch · CLS cosine", cls_sim, "coolwarm"), ("last-block CLS attention (head mean)", head_attn, "inferno"), ("pre-norm patch token norm", patch_norm, "viridis")]
    for ax, (title, heat, cmap) in zip(axes[1:4], overlays):
        ax.imshow(img)
        im = ax.imshow(heat.reshape(grid, grid).cpu(), cmap=cmap, alpha=0.6, extent=extent, interpolation="nearest")
        fig.colorbar(im, ax=ax, fraction=0.046)
        ax.set_title(title)
    axes[4].imshow(pca_rgb, interpolation="nearest")
    axes[4].set_title("patch PCA → RGB (dataset basis)")
    for ax in axes[:5]:
        ax.axis("off")
    axes[5].scatter(bg_pcs[:, 0], bg_pcs[:, 1], s=1, c="lightgray", alpha=0.3, label="val patches")
    sc = axes[5].scatter(patch_pcs[:, 0], patch_pcs[:, 1], s=10, c=cls_sim.cpu(), cmap="coolwarm", label="this tile")
    axes[5].scatter(cls_pc[:, 0], cls_pc[:, 1], s=200, marker="*", c="k", label="CLS")
    fig.colorbar(sc, ax=axes[5], fraction=0.046, label="patch · CLS cos")
    axes[5].set(title="Patch tokens in PC1/PC2", xlabel="PC1", ylabel="PC2")
    axes[5].legend(loc="upper right", markerscale=2)
    # Where each head's CLS attention goes; a tall register band means dim patch maps are expected, not a failure.
    labels = [f"h{i}" for i in range(len(attn_mass))] + ["mean"]
    mass = torch.cat([attn_mass, attn_mass.mean(0, keepdim=True)]).numpy()
    bottom = np.zeros(len(labels))
    for j, name in enumerate(["self", "registers", "patches"]):
        axes[6].bar(labels, mass[:, j], bottom=bottom, label=name)
        bottom += mass[:, j]
    axes[6].set(title="CLS attention mass by token type", ylabel="attention mass", ylim=(0, 1))
    axes[6].legend(loc="upper right")
    top = head_attn.topk(5).indices
    axes[7].scatter(patch_norm.cpu(), head_attn.cpu(), s=10, alpha=0.6, label="patches")
    axes[7].scatter(patch_norm[top].cpu(), head_attn[top].cpu(), s=40, c="r", label="top-5 attention")
    for v in reg_norm.tolist():
        axes[7].axvline(v, color="k", ls=":", alpha=0.5)
    axes[7].set(title="CLS attention vs pre-norm norm (dotted = registers)", xlabel="pre-norm token norm", ylabel="CLS attention (head mean)", yscale="log")
    axes[7].legend(loc="upper left")
    plt.tight_layout()
    plt.show()
    return stats


show_patch_tokens(model, val_ds, emb, idx=1000)

## Slide-balanced CLS structure
Random tiles give few same-slide pairs, so sample `N_SLIDES` val slides × `TILES_PER_SLIDE` tiles each. For every principal component of the CLS cloud, eta² is the fraction of its variance explained by slide identity (biology + stain/scanner) and by tissue source site (the `TCGA-XX` code; closest to a centre batch effect). High-variance PCs with high eta² are the candidates to flatten; high-variance PCs with chance-level eta² are likely signal.

In [ ]:
N_SLIDES, TILES_PER_SLIDE = 128, 16

# Slide stem and tissue source site for every val tile, read from the parquet path column only (no JPEG decode).
rows_by_shard = {}
for i, shard in enumerate(val_ds.shard_of):
    rows_by_shard.setdefault(int(shard), []).append(i)
val_paths = [None] * len(val_ds)
for shard, idxs in rows_by_shard.items():
    col = pq.read_table(str(val_ds.shards[shard]), columns=["path"], memory_map=True)["path"].to_pylist()
    for i in idxs:
        val_paths[i] = col[val_ds.row_of[i]]
val_slide = np.array([p.split("/", 1)[0] for p in val_paths])
val_site = np.array([p.split("-")[1] for p in val_paths])

slide_tiles = {}
for i, slide in enumerate(val_slide):
    slide_tiles.setdefault(slide, []).append(i)
eligible = sorted(s for s, tiles in slide_tiles.items() if len(tiles) >= TILES_PER_SLIDE)
rng = random.Random(SEED)
balanced_idx = [i for s in rng.sample(eligible, N_SLIDES) for i in rng.sample(slide_tiles[s], TILES_PER_SLIDE)]
print(f"val slides: {len(slide_tiles):,}  with >= {TILES_PER_SLIDE} tiles: {len(eligible):,}  sites: {len(set(val_site)):,}  balanced tiles: {len(balanced_idx):,}")
emb_bal = embed(model, val_ds, balanced_idx)

In [ ]:
# Per-PC eta² of CLS coordinates for slide and site labels, with a shuffled-label chance floor; variance-weighted eta² is the share of all CLS variance each label explains.
@torch.no_grad()
def slide_variance(emb, n_pcs=64, highlight=8):
    z = emb["cls"].to(device).double()
    z = z - z.mean(0)
    _, S, Vh = torch.linalg.svd(z, full_matrices=False)
    scores = z @ Vh.T
    var = S ** 2 / (S ** 2).sum()
    idx = emb["idx"].numpy()
    groups = {"slide": val_slide[idx], "site": val_site[idx]}
    perm = np.random.default_rng(SEED).permutation(len(idx))

    def eta2(labels):
        codes = torch.as_tensor(np.unique(labels, return_inverse=True)[1], device=device)
        sums = torch.zeros(int(codes.max()) + 1, scores.shape[1], dtype=scores.dtype, device=device).index_add_(0, codes, scores)
        return ((sums ** 2) / torch.bincount(codes)[:, None]).sum(0) / (scores ** 2).sum(0).clamp_min(1e-30)

    eta = {k: eta2(v) for k, v in groups.items()}
    chance = {k: eta2(v[perm]) for k, v in groups.items()}
    stats = {"n_tiles": len(idx), "n_slides": len(np.unique(groups["slide"])), "n_sites": len(np.unique(groups["site"]))}
    for k in groups:
        stats[f"{k}_share_of_variance"] = (var * eta[k]).sum().item()
        stats[f"{k}_share_of_top10_pc_variance"] = ((var[:10] * eta[k][:10]).sum() / var[:10].sum()).item()
        stats[f"{k}_chance_share"] = (var * chance[k]).sum().item()

    fig, axes = plt.subplots(1, 3, figsize=(20, 5))
    pcs = np.arange(n_pcs)
    for k, color in zip(groups, ["C0", "C1"]):
        axes[0].plot(pcs, eta[k][:n_pcs].cpu().numpy(), marker=".", color=color, label=f"{k} eta²")
        axes[0].axhline(chance[k][:n_pcs].mean().item(), color=color, ls="--", alpha=0.7, label=f"{k} chance")
    axes[0].set(title="Variance explained by label, per CLS PC", xlabel="PC", ylabel="eta²", ylim=(0, 1))
    axes[0].legend(loc="upper right")
    twin = axes[0].twinx()
    twin.bar(pcs, var[:n_pcs].cpu().numpy(), color="gray", alpha=0.25)
    twin.set(ylabel="PC variance fraction", yscale="log")
    # Highlight a few slides / the largest sites in PC1-PC2; the rest stay grey for context.
    xy = scores[:, :2].cpu().numpy()
    for ax, k in zip(axes[1:], groups):
        labels = groups[k]
        values, counts = np.unique(labels, return_counts=True)
        shown = values[np.argsort(-counts)][:highlight] if k == "site" else np.unique(labels)[:highlight]
        ax.scatter(xy[:, 0], xy[:, 1], s=3, c="lightgray", alpha=0.5)
        for j, v in enumerate(shown):
            m = labels == v
            ax.scatter(xy[m, 0], xy[m, 1], s=12, color=plt.cm.tab10(j % 10), label=v[:23])
        ax.set(title=f"CLS PC1/PC2 coloured by {k} ({len(shown)} shown)", xlabel="PC1", ylabel="PC2")
        ax.legend(fontsize=7, markerscale=1.5, loc="best")
    plt.tight_layout()
    plt.show()
    return stats


slide_variance(emb_bal)

In [ ]:
# Same-slide vs cross-slide cosine is well populated here (128 slides x 16 tiles x 15 = 30,720 same-slide pairs).
cls_spread(emb_bal)